# 20 — READ ground truth and frozen labels

This notebook freezes the roster, donor bank, folds, model scope, and
GO/NO-GO bar before any new model result. It computes only the two causal
ground truths. It does not run P1/P2/P3, controls, nulls, capability, tests,
lint, or an alpha sweep.

In [ ]:
from pathlib import Path
import hashlib
import json
import subprocess

from src.read_validation import (
    READ_VALIDATION_PROTOCOL,
    READ_VALIDATION_PROTOCOL_SHA256,
    assign_balanced_folds,
    clean_dashboard_roster,
    read_json,
    strict_engine_roster,
    write_json,
)

ROOT = Path.cwd()
assert ROOT.name == 'j-space-thoughts'
RAW_DIR = ROOT / 'data/raw/v5'
RAW_DIR.mkdir(parents=True, exist_ok=True)

def command(*args):
    return subprocess.check_output(args, text=True).strip()

hf_path = command('bash', '-lc', 'command -v hf')
hf_identity = command('hf', 'auth', 'whoami')
gpu_csv = command(
    'nvidia-smi',
    '--query-gpu=name,memory.total,memory.free',
    '--format=csv,noheader,nounits',
)
gpu_name, total_mib, free_mib = [part.strip() for part in gpu_csv.split(',')]
disk_line = command('df', '-BG', '--output=size,used,avail,pcent', str(Path.home() / '.cache/huggingface')).splitlines()[-1].split()
preflight = {
    'hf_path': hf_path,
    'hf_identity': hf_identity,
    'gpu': {'name': gpu_name, 'memory_total_mib': int(total_mib), 'memory_free_mib': int(free_mib)},
    'disk': {'size_gib': int(disk_line[0][:-1]), 'used_gib': int(disk_line[1][:-1]), 'free_gib': int(disk_line[2][:-1]), 'use_percent': disk_line[3]},
    'qwen3_dry_run_weight_sizes': {
        'Qwen/Qwen3-32B': 'about 66 GiB',
        'Qwen/Qwen3-30B-A3B-Instruct-2507': 'about 61 GiB',
        'Qwen/Qwen3-14B': 'about 30 GiB',
    },
    'scale_decision': {
        'status': 'SKIPPED_NO_COMPARABLE_QWEN3_INSTRUMENT',
        'reasons': [
            '32B and 30B bf16 weights exceed the 38 GiB free filesystem observed before outcomes.',
            '14B would leave inadequate artifact headroom.',
            'No Qwen3 model has a validated compatible published J-Lens/direction artifact in this repository.',
            'Building a new lens is out of scope and 4-bit is incompatible with the required bf16 derivatives.',
        ],
    },
}

metrics_path = ROOT / 'results/metrics.json'
metrics = read_json(metrics_path)
raw02 = read_json(ROOT / 'data/raw/02_twohop_qwen2.5-7b.json')
engines = strict_engine_roster(raw02)
assert engines['n_primary'] == engines['n_retained'] == 155

v2_rows = {
    row['key']: row
    for row in metrics['repair_v2']['stage2_recalibration']['g_pos']['rows']
}
v4_rows = {
    row['key']: row
    for row in metrics['calibration_v4']['stage10']['narration']['rows']
}
dashboard_inputs = []
for key, row in sorted(v2_rows.items()):
    sequence_length = len(next(iter(row['automatic_read']['attribution']['read_by_layer_position'].values())))
    dashboard_inputs.append({
        'row_id': key,
        'key': key,
        'language': row['category'],
        'sequence_length': sequence_length,
        'carrying_positions': v4_rows[key]['auto_mask'],
        'clean_capable': bool(row['automatic_clean_metric'] > 0.0),
    })
dashboards = clean_dashboard_roster(dashboard_inputs)
assert dashboards['n'] == 8 and dashboards['all_rows_retained']

fold_manifest = assign_balanced_folds([*engines['rows'], *dashboards['rows']])
assert len(fold_manifest['rows']) == 163
assert len({row['concept_id'] for row in fold_manifest['rows']}) == 79
assert fold_manifest['row_counts_by_label_fold']['0'] == [2, 2, 2, 2]

protocol_manifest = {
    'schema_version': 'read-go-no-go-preregistered-manifest-v1',
    'protocol': READ_VALIDATION_PROTOCOL,
    'protocol_sha256': READ_VALIDATION_PROTOCOL_SHA256,
    'preflight': preflight,
    'engine_roster': {key: value for key, value in engines.items() if key != 'rows'},
    'dashboard_roster': {key: value for key, value in dashboards.items() if key != 'rows'},
    'fold_manifest': fold_manifest,
    'counts': {
        'cases': 163,
        'engine_cases': 155,
        'dashboard_cases': 8,
        'semantic_concepts': 79,
        'engine_concepts': 75,
        'dashboard_concepts': 4,
    },
    'result_fields_absent_at_freeze': ['ground_truth_A', 'ground_truth_B', 'R1', 'R2', 'R3', 'AUC', 'DECISION'],
}
manifest_artifact = write_json(RAW_DIR / '20_protocol_manifest.json', protocol_manifest)

existing = metrics.get('read_validation_v5')
if existing is not None:
    assert existing['protocol_sha256'] == READ_VALIDATION_PROTOCOL_SHA256
metrics['read_validation_v5'] = {
    'schema_version': 'read-go-no-go-v1',
    'status': 'RUNNING',
    'protocol': READ_VALIDATION_PROTOCOL,
    'protocol_sha256': READ_VALIDATION_PROTOCOL_SHA256,
    'preflight': preflight,
    'manifest_artifact': manifest_artifact,
    'counts': protocol_manifest['counts'],
    'stage20': {'status': 'RUNNING'},
    'stage21': {'status': 'PENDING'},
    'stage22': {'status': 'PENDING'},
    'decision': 'PENDING',
    'hypothesis_status': 'NOT_TESTED',
}
write_json(metrics_path, metrics)
print(json.dumps({'preflight': preflight, 'counts': protocol_manifest['counts'], 'protocol_sha256': READ_VALIDATION_PROTOCOL_SHA256, 'fold_counts': fold_manifest['row_counts_by_label_fold']}, indent=2))

In [ ]:
import gc
import torch

from src.jlens_iface import load_published_lens
from src.model_utils import load_model
from src.v3_alpha_sweep import _build_bank

bundle = load_model(READ_VALIDATION_PROTOCOL['model']['id'])
lens = load_published_lens(bundle.model_id)
assert bundle.revision == READ_VALIDATION_PROTOCOL['model']['revision']
assert next(bundle.hf_model.parameters()).dtype == torch.bfloat16

source_ids = {int(row['source_token_id']) for row in engines['rows']}
foil_ids = {int(row['foil_concept_token_id']) for row in engines['rows']}
language_ids = {int(row['language_direction_token_id']) for row in v2_rows.values()}
english_ids = {int(row['english_direction_token_id']) for row in v2_rows.values()}
assert len(english_ids) == 1
token_ids = source_ids | foil_ids | language_ids | english_ids
bank = _build_bank(bundle, lens, token_ids, READ_VALIDATION_PROTOCOL['workspace_layers'])
assert all(set(range(13, 28)).issubset(bank[token_id]) for token_id in token_ids)
bank_cpu = {
    int(token_id): {int(layer): vector.detach().cpu() for layer, vector in layers.items()}
    for token_id, layers in bank.items()
}
direction_bank_path = RAW_DIR / 'qwen2.5-7b_read_validation_directions.pt'
torch.save({'protocol_sha256': READ_VALIDATION_PROTOCOL_SHA256, 'bank': bank_cpu}, direction_bank_path)
print({'model': bundle.model_id, 'revision': bundle.revision, 'dtype': str(next(bundle.hf_model.parameters()).dtype), 'direction_tokens': len(bank), 'direction_cache_bytes': direction_bank_path.stat().st_size})

In [ ]:
from jlens.hooks import ActivationRecorder

from src.interventions import forward_logits
from src.v3_alpha_sweep import _mask_for_prompt

device = next(bundle.hf_model.parameters()).device
workspace_layers = list(READ_VALIDATION_PROTOCOL['workspace_layers'])
raw_primary = {
    row['name']: row
    for row in raw02['rows']
    if row.get('direction_method') == 'jlens_raw_wu_j'
}
manifest_rows = {row['row_id']: row for row in fold_manifest['rows']}
clean_cache = {}

def capture_clean(input_ids):
    with torch.no_grad(), ActivationRecorder(bundle.lens_model.layers, at=workspace_layers) as recorder:
        bundle.hf_model(input_ids=input_ids, use_cache=False)
    return {int(layer): value.detach().cpu() for layer, value in recorder.activations.items()}

for index, row in enumerate(engines['rows'], start=1):
    raw = raw_primary[row['row_id']]
    input_ids = torch.tensor([raw['prompt_token_ids']], dtype=torch.long, device=device)
    assert bundle.lens_model.encode(row['prompt']).detach().cpu().tolist()[0] == raw['prompt_token_ids']
    mask, _ = _mask_for_prompt(bundle, lens, row['prompt'], int(row['source_token_id']), workspace_layers)
    clean_cache[row['row_id']] = {
        'input_ids': input_ids.detach().cpu(),
        'residuals': capture_clean(input_ids),
        'carrying_mask': mask,
        'metric_payload': {
            'type': 'next_token_difference',
            'target_token_id': int(row['clean_target_token_id']),
            'foil_token_id': int(row['counterfactual_target_token_id']),
        },
    }
    if index % 10 == 0 or index == len(engines['rows']):
        print(f'engine clean cache {index}/{len(engines["rows"])}')

gpos = metrics['repair_v2']['stage2_recalibration']['g_pos']
families = gpos['token_family_manifest']
for key, prior in sorted(v2_rows.items()):
    prompt_ids = bundle.lens_model.encode(prior['automatic_prompt'])
    continuation = torch.tensor([prior['frozen_continuation']['token_ids']], dtype=prompt_ids.dtype, device=prompt_ids.device)
    full_ids = torch.cat([prompt_ids, continuation], dim=1)
    expected_length = int(manifest_rows[key]['sequence_length'])
    assert full_ids.shape[1] == expected_length
    clean_cache[key] = {
        'input_ids': full_ids.detach().cpu(),
        'residuals': capture_clean(full_ids),
        'carrying_mask': {
            'rule': 'frozen v4 source J-Lens rank<=10 union',
            'rank_threshold': 10,
            'positions': list(manifest_rows[key]['carrying_positions']),
            'sequence_length': expected_length,
        },
        'metric_payload': {
            'type': 'language_mass',
            'score_positions': list(prior['frozen_continuation']['score_positions']),
            'source_token_ids': list(families[prior['category']]['token_ids']),
            'english_token_ids': list(families['English']['token_ids']),
        },
        'source_token_id': int(prior['language_direction_token_id']),
        'foil_concept_token_id': int(prior['english_direction_token_id']),
    }
print('dashboard clean cache', len(v2_rows))

clean_cache_path = RAW_DIR / '20_clean_residual_cache.pt'
torch.save({'protocol_sha256': READ_VALIDATION_PROTOCOL_SHA256, 'rows': clean_cache}, clean_cache_path)
print({'clean_cache_rows': len(clean_cache), 'bytes': clean_cache_path.stat().st_size})

In [ ]:
import math

from src.interventions import forward_logits
from src.read_validation import (
    coordinate_resampling_edits,
    coordinate_resampling_effect,
    label_verification_report,
    masked_source_to_foil_effect,
)
from src.v2_recalibration import language_mass_metric

def metric_from_payload(payload):
    if payload['type'] == 'next_token_difference':
        target = int(payload['target_token_id'])
        foil = int(payload['foil_token_id'])
        return lambda logits: logits[0, -1, target].float() - logits[0, -1, foil].float()
    positions = list(payload['score_positions'])
    source_tokens = list(payload['source_token_ids'])
    english_tokens = list(payload['english_token_ids'])
    return lambda logits: language_mass_metric(logits, positions, source_tokens, english_tokens)

checkpoint_path = RAW_DIR / '20_ground_truth_checkpoint.json'
completed = {}
if checkpoint_path.exists():
    checkpoint = read_json(checkpoint_path)
    if checkpoint.get('protocol_sha256') == READ_VALIDATION_PROTOCOL_SHA256:
        completed = {row['row_id']: row for row in checkpoint.get('rows', [])}

for index, manifest_row in enumerate(fold_manifest['rows'], start=1):
    row_id = manifest_row['row_id']
    if row_id in completed:
        continue
    cached = clean_cache[row_id]
    input_ids = cached['input_ids'].to(device)
    clean_logits = forward_logits(bundle.hf_model, input_ids)
    source_id = int(manifest_row.get('source_token_id', clean_cache[row_id].get('source_token_id')))
    foil_id = int(manifest_row.get('foil_concept_token_id', clean_cache[row_id].get('foil_concept_token_id')))
    directions = {layer: bank[source_id][layer] for layer in workspace_layers}
    foil_directions = {layer: bank[foil_id][layer] for layer in workspace_layers}
    positions = list(cached['carrying_mask']['positions'])
    metric_fn = metric_from_payload(cached['metric_payload'])
    a_manifest = None
    if not positions:
        ground_a = {'status': 'UNDEFINED_EMPTY_CARRYING_MASK', 'causal_abs': None}
    elif manifest_row.get('donor_row_id') not in clean_cache:
        ground_a = {'status': 'UNDEFINED_DONOR_NOT_CACHED', 'causal_abs': None}
    else:
        try:
            a_edits, a_manifest = coordinate_resampling_edits(
                cached['residuals'],
                clean_cache[manifest_row['donor_row_id']]['residuals'],
                directions,
                positions,
            )
            ground_a = coordinate_resampling_effect(
                bundle.hf_model,
                bundle.lens_model.layers,
                input_ids,
                metric_fn,
                a_edits,
                clean_logits=clean_logits,
            )
        except (IndexError, ValueError) as error:
            ground_a = {'status': 'UNDEFINED_FROZEN_DONOR_GEOMETRY', 'causal_abs': None, 'error': str(error)}
    if not positions:
        ground_b = {'status': 'UNDEFINED_EMPTY_CARRYING_MASK', 'causal_abs': None}
    else:
        ground_b = masked_source_to_foil_effect(
            bundle.hf_model,
            bundle.lens_model.layers,
            input_ids,
            metric_fn,
            cached['residuals'],
            directions,
            foil_directions,
            positions,
            clean_logits=clean_logits,
        )
    completed[row_id] = {
        **manifest_row,
        'sequence_length': int(input_ids.shape[1]),
        'input_token_ids': input_ids.detach().cpu().tolist()[0],
        'carrying_mask': cached['carrying_mask'],
        'source_token_id': source_id,
        'foil_concept_token_id': foil_id,
        'metric_payload': cached['metric_payload'],
        'ground_truth_A': ground_a,
        'ground_truth_B': ground_b,
        'A_coordinate_manifest': a_manifest,
    }
    write_json(checkpoint_path, {'protocol_sha256': READ_VALIDATION_PROTOCOL_SHA256, 'rows': [completed[key] for key in sorted(completed)]})
    print(f'ground truth {index}/{len(fold_manifest["rows"])} {row_id}: A={ground_a.get("causal_abs")} B={ground_b.get("causal_abs")}')
    del clean_logits
    gc.collect()
    torch.cuda.empty_cache()

ground_rows = [completed[key] for key in sorted(completed)]
assert len(ground_rows) == 163
label_verification = label_verification_report(ground_rows)
print(json.dumps({
    'rows': len(ground_rows),
    'A_defined': sum(row['ground_truth_A'].get('causal_abs') is not None for row in ground_rows),
    'B_defined': sum(row['ground_truth_B'].get('causal_abs') is not None for row in ground_rows),
    'label_verification': {key: label_verification[key] for key in ('status', 'n_rows', 'n_failures')},
}, indent=2))

In [ ]:
import numpy as np
from scipy.stats import spearmanr

raw20 = {
    'schema_version': 'read-ground-truth-v1',
    'protocol_sha256': READ_VALIDATION_PROTOCOL_SHA256,
    'protocol': READ_VALIDATION_PROTOCOL,
    'manifest_artifact': manifest_artifact,
    'preflight': preflight,
    'model': {'id': bundle.model_id, 'revision': bundle.revision, 'dtype': str(next(bundle.hf_model.parameters()).dtype)},
    'direction_bank_cache': str(direction_bank_path.relative_to(ROOT)),
    'label_verification': label_verification,
    'rows': ground_rows,
}
raw20_artifact = write_json(RAW_DIR / '20_read_ground_truth.json', raw20)

finite = [
    row for row in ground_rows
    if row['ground_truth_A'].get('causal_abs') is not None
    and row['ground_truth_B'].get('causal_abs') is not None
]
a_values = np.asarray([row['ground_truth_A']['causal_abs'] for row in finite], dtype=float)
b_values = np.asarray([row['ground_truth_B']['causal_abs'] for row in finite], dtype=float)
correlation = {
    'n_cases': len(finite),
    'pearson_r': float(np.corrcoef(a_values, b_values)[0, 1]) if len(finite) > 1 else None,
    'spearman_rho': float(spearmanr(a_values, b_values).statistic) if len(finite) > 1 else None,
}

metrics = read_json(metrics_path)
v5 = metrics['read_validation_v5']
assert v5['protocol_sha256'] == READ_VALIDATION_PROTOCOL_SHA256
v5['stage20'] = {
    'status': 'COMPLETE',
    'raw_artifact': raw20_artifact,
    'ground_truth_correlation_case_level': correlation,
    'label_verification': {key: value for key, value in label_verification.items() if key != 'rows'},
    'counts': {
        'cases': len(ground_rows),
        'A_defined': sum(row['ground_truth_A'].get('causal_abs') is not None for row in ground_rows),
        'B_defined': sum(row['ground_truth_B'].get('causal_abs') is not None for row in ground_rows),
    },
    'rows': [
        {
            'row_id': row['row_id'], 'concept_id': row['concept_id'], 'fold_group': row['fold_group'],
            'fold': row['fold'], 'label': row['label'], 'class_name': row['class_name'],
            'clean_capable': row.get('clean_capable'), 'donor_row_id': row.get('donor_row_id'),
            'A_causal_abs': row['ground_truth_A'].get('causal_abs'),
            'B_causal_abs': row['ground_truth_B'].get('causal_abs'),
            'A_status': row['ground_truth_A']['status'], 'B_status': row['ground_truth_B']['status'],
        }
        for row in ground_rows
    ],
}
v5['stage21'] = {'status': 'PENDING'}
v5['stage22'] = {'status': 'PENDING'}
write_json(metrics_path, metrics)

report = f'''# READ Go/No-Go validation — ground truth complete

## Preflight

- GPU: {preflight['gpu']['name']}; {preflight['gpu']['memory_total_mib']} MiB total, {preflight['gpu']['memory_free_mib']} MiB free.
- HF filesystem free: {preflight['disk']['free_gib']} GiB.
- Qwen3 scale arm: **SKIPPED** — no comparable validated Qwen3 J-Lens instrument; 32B/30B also exceed disk.

## Frozen scope

Validation only. P1/P2/P3 are not tested. Exactly R1/R2/R3 and seven fixed candidate summaries are permitted. Protocol SHA-256: `{READ_VALIDATION_PROTOCOL_SHA256}`.

## Notebook 20

- 163 cases: 155 engines and all 8 dashboards.
- 79 semantic concepts: 75 engines and 4 dashboard languages.
- Coordinate-resampling A defined for {v5['stage20']['counts']['A_defined']}/163; fixed masked alpha=1.5 source-to-foil B defined for {v5['stage20']['counts']['B_defined']}/163.
- A/B case-level correlation: Pearson r={correlation['pearson_r']}, Spearman rho={correlation['spearman_rho']} (N={correlation['n_cases']}).
- Declared label verification: **{label_verification['status']}**, failures={label_verification['n_failures']} (failures retained).

No READ score or AUC has been inspected yet. Notebook 21 is next.
'''
(ROOT / 'results/RESULTS.md').write_text(report, encoding='utf-8')

from src.model_utils import release_model
del clean_cache, bank, bank_cpu, lens
release_model(bundle)
print(json.dumps({'raw20': raw20_artifact, 'correlation': correlation, 'label_verification': label_verification['status']}, indent=2))